# Video Extraction & Analysis Tool

Analyze videos using **Claude AI vision** and **Whisper transcription**.

### Steps
1. Run cells 1-3 to install everything (~3 min)
2. Paste your Anthropic API key in cell 4
3. Run cell 5 to analyze a video

> **Note:** YouTube and most video hosts block Colab IPs. Use a local file or upload your own video in cell 5.


In [ ]:
# Cell 1 - Install ffmpeg
!apt-get install -y ffmpeg 2>&1 | tail -3
print('ffmpeg ready.')

In [ ]:
# Cell 2 - Clone repo and install Python dependencies
!git clone https://github.com/codedaddylive/claude-code-repo /content/video-tool 2>&1 | tail -3
%cd /content/video-tool
!pip install -r requirements.txt -q
print('All dependencies installed.')

In [ ]:
# Cell 3 - Run integration test (no API key needed)
import sys
sys.path.insert(0, '/content/video-tool')
!python3 tests/integration_test.py

In [ ]:
# Cell 4 - Set your Anthropic API key
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # <-- paste your real key here

# Or use Colab Secrets (key icon in left sidebar, name it ANTHROPIC_API_KEY):
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

print('API key set:', 'yes' if os.environ.get('ANTHROPIC_API_KEY','').startswith('sk-') else 'NOT SET')

In [ ]:
# Cell 5 - Analyze a video
#
# OPTION A (default): generate a 20s test video using ffmpeg
# OPTION B: upload your own file
#   from google.colab import files
#   uploaded = files.upload()  # pick a .mp4 from your computer
#   VIDEO_PATH = '/content/' + list(uploaded.keys())[0]

import subprocess, sys
sys.path.insert(0, '/content/video-tool')

# Generate test video (colorful countdown with audio)
VIDEO_PATH = '/content/test_video.mp4'
subprocess.run([
    'ffmpeg', '-y',
    '-f', 'lavfi', '-i', 'testsrc2=duration=20:size=640x360:rate=25',
    '-f', 'lavfi', '-i', 'sine=frequency=440:duration=20',
    '-c:v', 'libx264', '-c:a', 'aac', '-shortest', VIDEO_PATH
], capture_output=True, check=True)
print(f'Test video created: {VIDEO_PATH}')

# Run analysis
!python3 cli.py analyze "{VIDEO_PATH}" \
    --max-frames 6 \
    --interval 3 \
    --output /content/result.json

import json
with open('/content/result.json') as f:
    result = json.load(f)

print(f"\nDuration: {result.get('duration_sec', '?'):.1f}s")
print(f"Frames analysed: {result.get('frame_count', 0)}")
if result.get('visual_summary'):
    print(f"\n--- Visual Summary ---\n{result['visual_summary']}")
if result.get('transcription') and result['transcription']['full_text']:
    print(f"\n--- Transcript ---\n{result['transcription']['full_text'][:500]}")

In [ ]:
# Cell 6 - Upload YOUR OWN video and analyze it
from google.colab import files
import sys, os, json
sys.path.insert(0, '/content/video-tool')
os.chdir('/content/video-tool')

print('Select a video file from your computer...')
uploaded = files.upload()
video_path = '/content/' + list(uploaded.keys())[0]
print(f'Uploaded: {video_path}')

!python3 cli.py analyze "{video_path}" \
    --max-frames 8 \
    --interval 5 \
    --output /content/my_result.json

with open('/content/my_result.json') as f:
    result = json.load(f)

print(f"\nDuration: {result.get('duration_sec', '?'):.1f}s | Frames: {result.get('frame_count', 0)}")
if result.get('visual_summary'):
    print(f"\n--- Visual Summary ---\n{result['visual_summary']}")
if result.get('transcription') and result['transcription']['full_text']:
    print(f"\n--- Transcript ---\n{result['transcription']['full_text'][:1000]}")

In [ ]:
# Cell 7 - Show keyframe descriptions from last result
import json
with open('/content/result.json') as f:
    result = json.load(f)
for i, desc in enumerate(result.get('keyframe_descriptions', [])):
    print(f"Frame {i}: {desc}")

In [ ]:
# Cell 8 - Full integration test WITH Claude vision (requires API key in cell 4)
!python3 tests/integration_test.py